In [52]:
import sys
sys.path.append("/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR")

# Experimentation with PyOD Models

This notebook evaluates PyOD models on two UCI datasets that are more suitable for anomaly detection: `shuttle` and `arrhythmia`. The goal is to load the datasets from the RADAR static dataset module, reframe them as anomaly-detection benchmarks, and compare several PyOD models using label-based and score-based metrics.

## Import Required Libraries

Import the necessary libraries for data loading, preprocessing, model training, and visualization.

In [59]:
# Import Required Libraries
import importlib

import numpy as np
import pandas as pd

from RADAR.static_data.algorithms import pyod
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

## UCI Experiment: Shuttle and Arrhythmia

This benchmark uses two UCI datasets that are more appropriate for anomaly detection than the earlier classification-style datasets.

The anomaly-benchmark preparation logic now lives in `RADAR.static_data.anomaly_dataset_utils`, so the notebook only configures which datasets and labels to use.

In [54]:
import RADAR.static_data.anomaly_dataset_utils as anomaly_dataset_utils

anomaly_dataset_utils = importlib.reload(anomaly_dataset_utils)

uci_dataset_configs = {
    "shuttle": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="shuttle",
        normal_label=1,
        target_test_contamination=0.1,
        max_train_normals=8000,
        max_test_size=5000,
    ),
    "arrhythmia": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="arrhythmia",
        normal_label=1,
        target_test_contamination=0.1,
    ),
}

In [55]:
uci_summary_rows = []
for dataset_name, config in uci_dataset_configs.items():
    uci_summary_rows.append(
        {
            "dataset": dataset_name,
            "samples": config["n_samples"],
            "features": config["n_features"],
            "original_anomaly_ratio": round(config["original_positive_ratio"], 4),
            "benchmark_test_contamination": round(
                config["benchmark_test_positive_ratio"], 4
            ),
            "train_normals_used": config["train_normals"],
            "test_normals": config["test_normals"],
            "test_anomalies": config["test_anomalies"],
        }
    )

uci_summary_df = pd.DataFrame(uci_summary_rows)
display(uci_summary_df)

,dataset,samples,features,original_anomaly_ratio,benchmark_test_contamination,train_normals_used,test_normals,test_anomalies
0,shuttle,58000,7,0.214,0.1036,8000,4482,518
1,arrhythmia,452,279,0.458,0.0926,196,49,5


### PyOD Experiment on Shuttle and Arrhythmia

For this additional comparison we keep the same anomaly-detection framing, but use a more scalable list of models to avoid extremely slow runs on `shuttle`.

In [57]:
uci_pyod_models = [
    {"algorithm_": "cblof"},
    {"algorithm_": "iforest", "random_state": 42},
    {"algorithm_": "knn", "n_neighbors": 5},
    {"algorithm_": "hbos"},
    {"algorithm_": "ocsvm"},
    {"algorithm_": "lof", "n_neighbors": 5},
]

uci_results = []

for dataset_name, config in uci_dataset_configs.items():
    print(f"\nDataset: {dataset_name}")
    print(
        f"Training with normal-only samples: {config['train_normals']} | "
        f"Benchmark contamination: {config['benchmark_test_positive_ratio']:.3f}"
    )

    for model_params in uci_pyod_models:
        model_kwargs = {
            **model_params,
            "contamination": config["benchmark_test_positive_ratio"],
        }

        model = pyod.PyodAnomalyDetection(**model_kwargs)
        model.fit(config["X_train"])
        predictions = np.asarray(model.predict(config["X_test"])).astype(int).ravel()
        scores = np.asarray(model.decision_function(config["X_test"])).ravel()

        accuracy = metrics_module.metric_accuracy(config["y_test"], predictions) / 100
        precision = metrics_module.metric_precision(config["y_test"], predictions)
        recall = metrics_module.metric_recall(config["y_test"], predictions)
        f1 = metrics_module.metric_F1score(config["y_test"], predictions)

        finite_scores = np.isfinite(scores)
        if finite_scores.all():
            roc_auc = metrics_module.metric_AUC_ROC_scores(config["y_test"], scores)
            pr_auc = metrics_module.metric_PR_AUC(config["y_test"], scores)
            score_note = ""
        else:
            roc_auc = np.nan
            pr_auc = np.nan
            score_note = " | score metrics skipped (NaN decision scores)"

        print(f"\nModel: {model_params['algorithm_']}{score_note}")
        metrics_module.print_metrics(["Accuracy", "Precision", "Recall", "F1"], config["y_test"], predictions)
        if finite_scores.all():
            print(f"ROC AUC (scores): {roc_auc:.3f}")
            print(f"PR AUC (scores): {pr_auc:.3f}")

        uci_results.append(
            {
                "dataset": dataset_name,
                "algorithm": model_params["algorithm_"],
                "contamination": round(config["benchmark_test_positive_ratio"], 4),
                "accuracy": round(accuracy, 4),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "roc_auc_scores": round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
                "pr_auc_scores": round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
            }
        )

uci_results_df = pd.DataFrame(uci_results).sort_values(
    ["dataset", "pr_auc_scores", "roc_auc_scores"],
    ascending=[True, False, False],
    na_position="last",
).reset_index(drop=True)

display(uci_results_df)


Dataset: shuttle
Training with normal-only samples: 8000 | Benchmark contamination: 0.104

Model: cblof
Accuracy: 89.280%
Precision: 0.490
Recall: 0.820
F1 Score: 0.613
ROC AUC (scores): 0.957
PR AUC (scores): 0.763

Model: iforest
Accuracy: 86.840%
Precision: 0.412
Recall: 0.633
F1 Score: 0.499
ROC AUC (scores): 0.883
PR AUC (scores): 0.623

Model: knn
Accuracy: 91.860%
Precision: 0.561
Recall: 0.992
F1 Score: 0.716
ROC AUC (scores): 0.993
PR AUC (scores): 0.950

Model: hbos
Accuracy: 86.600%
Precision: 0.395
Recall: 0.554
F1 Score: 0.461
ROC AUC (scores): 0.882
PR AUC (scores): 0.452

Model: ocsvm
Accuracy: 87.340%
Precision: 0.424
Recall: 0.618
F1 Score: 0.503
ROC AUC (scores): 0.909
PR AUC (scores): 0.685

Model: lof
Accuracy: 89.720%
Precision: 0.502
Recall: 0.992
F1 Score: 0.667
ROC AUC (scores): 0.988
PR AUC (scores): 0.858

Dataset: arrhythmia
Training with normal-only samples: 196 | Benchmark contamination: 0.093

Model: cblof
Accuracy: 77.778%
Precision: 0.267
Recall: 0.800


,dataset,algorithm,contamination,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores
0,arrhythmia,hbos,0.0926,0.7963,0.2500,0.6000,0.3529,0.8286,0.5417
1,arrhythmia,knn,0.0926,0.7593,0.2143,0.6000,0.3158,0.7755,0.4641
2,arrhythmia,cblof,0.0926,0.7778,0.2667,0.8000,0.4000,0.7837,0.4637
3,arrhythmia,ocsvm,0.0926,0.7222,0.1875,0.6000,0.2857,0.7510,0.4626
4,arrhythmia,iforest,0.0926,0.8889,0.4444,0.8000,0.5714,0.8327,0.4041
5,arrhythmia,lof,0.0926,0.8148,0.2222,0.4000,0.2857,0.7510,0.3796
6,shuttle,knn,0.1036,0.9186,0.5605,0.9923,0.7164,0.9933,0.9505
7,shuttle,lof,0.1036,0.8972,0.5020,0.9923,0.6667,0.9876,0.8582
8,shuttle,cblof,0.1036,0.8928,0.4896,0.8205,0.6133,0.9571,0.7632
9,shuttle,ocsvm,0.1036,0.8734,0.4238,0.6178,0.5027,0.9094,0.6846


### Timing Comparison: RADAR vs Direct PyOD

### How to Interpret the Timing Table

- `dataset`: dataset on which the comparison was run (`shuttle` or `arrhythmia`).
- `algorithm`: PyOD model being evaluated.
- `platform_time_s`: total execution time in seconds using the RADAR wrapper.
- `base_time_s`: total execution time in seconds using the direct PyOD class.
- `speedup_base_over_platform`: ratio `base_time_s / platform_time_s`. 
- `platform_roc_auc_scores`: score-based ROC-AUC obtained with the RADAR implementation.
- `base_roc_auc_scores`: score-based ROC-AUC obtained with the direct PyOD implementation.
- `roc_auc_diff`: difference `platform_roc_auc_scores - base_roc_auc_scores`. It shows whether the RADAR path preserves or changes ranking quality relative to direct PyOD.

### When Is One Better Than the Other?

- For **runtime**, RADAR is faster when `speedup_base_over_platform > 1` because the direct baseline took more time than the platform.
- If `speedup_base_over_platform < 1`, the direct PyOD implementation is faster.
- If `speedup_base_over_platform ≈ 1`, both approaches have very similar runtime.
- For **quality**, higher `platform_roc_auc_scores` or `base_roc_auc_scores` is better because ROC-AUC closer to `1.0` means better separation between normal samples and anomalies.
- If `roc_auc_diff > 0`, RADAR gives better score ranking quality than the direct baseline for that model and dataset.
- If `roc_auc_diff < 0`, the direct PyOD version gives better score ranking quality.
- Ideally, the preferred case is: `speedup_base_over_platform > 1` and `roc_auc_diff >= 0`, meaning RADAR is faster while keeping equal or better ROC-AUC.
- If one approach is faster but has lower ROC-AUC, then it is a trade-off between efficiency and detection quality.

In [71]:
import time
from statistics import mean
from tqdm import tqdm
 
direct_pyod_algorithms = {
    "cblof": CBLOF,
    "iforest": IForest,
    "knn": KNN,
    "hbos": HBOS,
    "ocsvm": OCSVM,
    "lof": LOF,
}
 
uci_timing_results = []
 
# Define the number of repetitions
num_repetitions = 30
 
for dataset_name, config in uci_dataset_configs.items():
    for model_params in uci_pyod_models:
        algorithm_name = model_params["algorithm_"]
        shared_kwargs = {
            key: value
            for key, value in model_params.items()
            if key != "algorithm_"
        }
 
        platform_model = pyod.PyodAnomalyDetection(
            algorithm_=algorithm_name,
            contamination=config["benchmark_test_positive_ratio"],
            **shared_kwargs,
        )
 
        # Perform multiple repetitions for platform execution
        platform_execution_times = []
        for _ in tqdm(range(num_repetitions), desc=f"Platform Execution Timing ({algorithm_name})"):
            start_time = time.time()
            platform_model.fit(config["X_train"])
            platform_execution_times.append(time.time() - start_time)
 
        direct_model_cls = direct_pyod_algorithms[algorithm_name]
        direct_model = direct_model_cls(
            contamination=config["benchmark_test_positive_ratio"],
            **shared_kwargs,
        )
 
        # Perform multiple repetitions for direct execution
        direct_execution_times = []
        for _ in tqdm(range(num_repetitions), desc=f"Direct Execution Timing ({algorithm_name})"):
            start_time = time.time()
            direct_model.fit(config["X_train"])
            direct_execution_times.append(time.time() - start_time)
 
        # Calculate average execution times
        average_platform_time = mean(platform_execution_times)
        average_direct_time = mean(direct_execution_times)
 
        platform_predictions = np.asarray(
            platform_model.predict(config["X_test"])
        ).astype(int).ravel()
        platform_scores = np.asarray(
            platform_model.decision_function(config["X_test"])
        ).ravel()
        platform_roc_auc = (
            metrics_module.metric_AUC_ROC_scores(config["y_test"], platform_scores)
            if np.isfinite(platform_scores).all()
            else np.nan
        )
 
        direct_predictions = np.asarray(
            direct_model.predict(config["X_test"])
        ).astype(int).ravel()
        direct_scores = np.asarray(
            direct_model.decision_function(config["X_test"])
        ).ravel()
        direct_roc_auc = (
            metrics_module.metric_AUC_ROC_scores(config["y_test"], direct_scores)
            if np.isfinite(direct_scores).all()
            else np.nan
        )
 
        # Calculate overhead
        overhead = average_platform_time - average_direct_time
 
        uci_timing_results.append(
            {
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "average_platform_time_s": round(average_platform_time, 4),
                "average_base_time_s": round(average_direct_time, 4),
                "overhead_s": round(overhead, 4),
                "speedup_base_over_platform": round(
                    average_direct_time / average_platform_time, 4
                ) if average_platform_time > 0 else np.nan,
                "platform_roc_auc_scores": round(float(platform_roc_auc), 4) if np.isfinite(platform_roc_auc) else np.nan,
                "base_roc_auc_scores": round(float(direct_roc_auc), 4) if np.isfinite(direct_roc_auc) else np.nan,
                "roc_auc_diff": round(float(platform_roc_auc - direct_roc_auc), 4)
                if np.isfinite(platform_roc_auc) and np.isfinite(direct_roc_auc)
                else np.nan,
            }
        )
 
uci_timing_df = pd.DataFrame(uci_timing_results).sort_values(
    ["dataset", "speedup_base_over_platform"],
    ascending=[True, False],
).reset_index(drop=True)
 
display(uci_timing_df)

Direct Execution Timing (lof): 100%|██████████| 30/30 [00:00<00:00, 392.64it/s]


,dataset,algorithm,average_platform_time_s,average_base_time_s,overhead_s,speedup_base_over_platform,platform_roc_auc_scores,base_roc_auc_scores,roc_auc_diff
0,arrhythmia,lof,0.0017,0.0025,-0.0008,1.4870,0.7510,0.7510,0.0000
1,arrhythmia,ocsvm,0.0052,0.0054,-0.0002,1.0347,0.7510,0.7510,0.0000
2,arrhythmia,hbos,0.0238,0.0240,-0.0002,1.0077,0.8286,0.8286,0.0000
3,arrhythmia,knn,0.0024,0.0023,0.0001,0.9670,0.7755,0.7755,0.0000
4,arrhythmia,cblof,0.0047,0.0044,0.0003,0.9338,0.8041,0.7714,0.0327
5,arrhythmia,iforest,0.1057,0.0920,0.0138,0.8699,0.8327,0.8327,0.0000
6,shuttle,lof,0.1687,0.1714,-0.0027,1.0159,0.9876,0.9876,0.0000
7,shuttle,knn,0.1685,0.1697,-0.0011,1.0066,0.9933,0.9933,0.0000
8,shuttle,ocsvm,1.8051,1.7953,0.0098,0.9946,0.9094,0.9094,0.0000
9,shuttle,iforest,0.1704,0.1691,0.0013,0.9923,0.8831,0.8831,0.0000


In [70]:
from pathlib import Path

results_dir = Path("/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results")
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / "uci_pyod_results.csv"
results_timing_path = results_dir / "uci_pyod_timing_results.csv"

uci_results_df.to_csv(results_main_path, index=False)
uci_timing_df.to_csv(results_timing_path, index=False)

print(f"Saved main results to: {results_main_path}")
print(f"Saved timing results to: {results_timing_path}")

Saved main results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_pyod_results.csv
Saved timing results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_pyod_timing_results.csv
